# Notebook 10: Three-Model Blend (05.7.1 + 06 + 07)

---

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.models import (
    DenseNet121_Weights,
    EfficientNet_B0_Weights,
    MobileNet_V2_Weights,
    ResNet18_Weights,
    densenet121,
    efficientnet_b0,
    mobilenet_v2,
    resnet18,
)

print("Imports OK")

## Step 2: Reproducibility and device

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## Step 3: Configuration

In [ ]:
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TRAIN_CSV = REPO / "data" / "train_val" / "train_val.csv"
TRAIN_IMG_DIR = REPO / "data" / "train_val" / "images"
TEST_IMG_DIR = REPO / "data" / "test_images"
PRED_DIR = REPO / "outputs" / "predictions"
CKPT_DIR = REPO / "outputs" / "checkpoints"

IMG_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 0
VAL_FRAC = 0.2
DROPOUT_0571 = 0.3
DROPOUT_07 = 0.3
WEIGHT_STEP = 0.05
MANUAL_W_06 = 0.20

CKPT_0571 = CKPT_DIR / "efnet_densenet_joint_ensemble_best.pt"
CKPT_07 = CKPT_DIR / "resnet_mobilenet_joint_ensemble_best.pt"
CSV_06 = max(PRED_DIR.glob("submission_06_oof_5050_efnet_densenet_*.csv"), default=None)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(f"IMG_SIZE   : {IMG_SIZE}")
print(f"BATCH_SIZE : {BATCH_SIZE}")
print(f"05.7.1 ckpt: {CKPT_0571} exists={CKPT_0571.exists()}")
print(f"07 ckpt    : {CKPT_07} exists={CKPT_07.exists()}")
print(f"06 csv     : {CSV_06}")
print(f"MANUAL_W_06: {MANUAL_W_06}")

## Step 4: Load labels

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df = df.rename(columns={
    "Image Index": "image_file",
    "Patient Age": "age",
    "Patient Sex": "sex",
    "Finding Labels": "finding",
})
df["label"] = (df["finding"] == "Cardiomegaly").astype(int)

print(df.head())
print("\nFinding counts:")
print(df["finding"].value_counts())
print(f"\nPositive rate: {df['label'].mean():.3f}")
print(f"Total         : {len(df)}")

## Step 5: Dataset

In [ ]:
class CardiomegalyDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, return_label=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.return_label = return_label

    def __len__(self):
        return len(self.df)

    def _load_image(self, fname):
        img = Image.open(self.img_dir / fname)
        if img.mode != "RGB":
            img = img.convert("RGB")
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_image(row["image_file"])
        if self.transform is not None:
            img = self.transform(img)
        if not self.return_label:
            return img, row["image_file"]
        return img, torch.tensor(row["label"], dtype=torch.float32)

## Step 6: Shared train / validation split

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=VAL_FRAC,
    stratify=df["label"],
    random_state=SEED,
)

print(f"Train: {len(train_df)}  (pos rate {train_df['label'].mean():.3f})")
print(f"Val  : {len(val_df)}  (pos rate {val_df['label'].mean():.3f})")

## Step 7: Transforms and loaders

In [ ]:
def build_transform(img_size):
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

val_tf = build_transform(IMG_SIZE)

train_eval_ds = CardiomegalyDataset(train_df, TRAIN_IMG_DIR, transform=val_tf)
val_ds = CardiomegalyDataset(val_df, TRAIN_IMG_DIR, transform=val_tf)
test_df = pd.DataFrame({
    "image_file": sorted(p.name for p in TEST_IMG_DIR.iterdir() if p.suffix.lower() == ".png")
})
test_ds = CardiomegalyDataset(test_df, TEST_IMG_DIR, transform=val_tf, return_label=False)

train_eval_loader = DataLoader(train_eval_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)

print(f"Train eval batches: {len(train_eval_loader)}")
print(f"Val        batches: {len(val_loader)}")
print(f"Test       batches: {len(test_loader)}")

## Step 8: Metrics helpers

In [ ]:
def sens_spec(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    pred = (y_prob >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    return sens, spec

def datathon_score(auroc, sens, spec):
    return 0.5 * auroc + 0.25 * sens + 0.25 * spec

def best_threshold_score(y_true, y_prob):
    auroc = roc_auc_score(y_true, y_prob)
    best_thr = 0.5
    best_score = -1.0
    for thr in np.linspace(0.01, 0.99, 199):
        sens, spec = sens_spec(y_true, y_prob, threshold=thr)
        score = datathon_score(auroc, sens, spec)
        if score > best_score:
            best_thr = float(thr)
            best_score = float(score)
    return best_thr, best_score

def metric_row(mode, y_true, y_prob, threshold=None):
    if threshold is None:
        threshold, score = best_threshold_score(y_true, y_prob)
    else:
        auroc = roc_auc_score(y_true, y_prob)
        sens, spec = sens_spec(y_true, y_prob, threshold=threshold)
        score = datathon_score(auroc, sens, spec)
    auroc = roc_auc_score(y_true, y_prob)
    sens, spec = sens_spec(y_true, y_prob, threshold=threshold)
    return {
        "mode": mode,
        "thr": float(threshold),
        "score": float(score),
        "auroc": float(auroc),
        "sens": float(sens),
        "spec": float(spec),
    }

## Step 9: Model definitions

In [ ]:
class EfficientDenseEnsemble(nn.Module):
    def __init__(self, dropout=0.3, efnet_weight=0.5):
        super().__init__()
        self.efnet = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        ef_in = self.efnet.classifier[1].in_features
        self.efnet.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(ef_in, 1))

        self.efnet_weight = efnet_weight

        self.dnet = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        dn_in = self.dnet.classifier.in_features
        self.dnet.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(dn_in, 1))

    def forward_logits(self, x):
        ef_logits = self.efnet(x).squeeze(1)
        dn_logits = self.dnet(x).squeeze(1)
        return ef_logits, dn_logits

    def forward(self, x, efnet_weight=None):
        ef_logits, dn_logits = self.forward_logits(x)
        weight = self.efnet_weight if efnet_weight is None else efnet_weight
        return weight * ef_logits + (1.0 - weight) * dn_logits


class ResNetMobileNetEnsemble(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.resnet = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        rn_in = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(rn_in, 1))

        self.mobilenet = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
        mb_in = self.mobilenet.classifier[1].in_features
        self.mobilenet.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(mb_in, 1))

    def forward_logits(self, x):
        res_logits = self.resnet(x).squeeze(1)
        mob_logits = self.mobilenet(x).squeeze(1)
        return res_logits, mob_logits

    def forward(self, x):
        res_logits, mob_logits = self.forward_logits(x)
        return (res_logits + mob_logits) / 2.0

## Step 10: Load 05.7.1 + 07 checkpoints and 06 test predictions

In [ ]:
def load_state(model, ckpt_path, device):
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    return model

@torch.no_grad()
def predict_probs_labeled(model, loader, device):
    model.eval()
    ys, ps = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        ys.extend(labels.numpy().tolist())
        ps.extend(probs.tolist())
    return np.array(ys), np.array(ps)

@torch.no_grad()
def predict_probs_named(model, loader, device):
    model.eval()
    names, ps = [], []
    for imgs, batch_names in loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        names.extend(batch_names)
        ps.extend(probs.tolist())
    return names, np.array(ps)

model_0571 = load_state(EfficientDenseEnsemble(dropout=DROPOUT_0571), CKPT_0571, device)
model_07 = load_state(ResNetMobileNetEnsemble(dropout=DROPOUT_07), CKPT_07, device)

train_y_0571, train_p_0571 = predict_probs_labeled(model_0571, train_eval_loader, device)
val_y_0571, val_p_0571 = predict_probs_labeled(model_0571, val_loader, device)
test_names_0571, test_p_0571 = predict_probs_named(model_0571, test_loader, device)

train_y_07, train_p_07 = predict_probs_labeled(model_07, train_eval_loader, device)
val_y_07, val_p_07 = predict_probs_labeled(model_07, val_loader, device)
test_names_07, test_p_07 = predict_probs_named(model_07, test_loader, device)

assert np.array_equal(train_y_0571, train_y_07)
assert np.array_equal(val_y_0571, val_y_07)
assert list(test_names_0571) == list(test_names_07)

train_y = train_y_0571
val_y = val_y_0571
test_names = test_names_0571

if CSV_06 is None:
    raise FileNotFoundError("Could not find a submission_06_oof_5050_efnet_densenet_*.csv file in outputs/predictions")

sub_06 = pd.read_csv(CSV_06).sort_values("image_file").reset_index(drop=True)
sub_test_names = sorted(test_names)
if sub_06["image_file"].tolist() != sub_test_names:
    raise ValueError("The 06 submission CSV does not align with the test image list.")

test_06_map = dict(zip(sub_06["image_file"], sub_06["prob"]))
test_p_06 = np.array([test_06_map[name] for name in test_names], dtype=np.float32)

base_rows = [
    metric_row("05.7.1_train", train_y, train_p_0571),
    metric_row("05.7.1_val", val_y, val_p_0571),
    metric_row("07_train", train_y, train_p_07),
    metric_row("07_val", val_y, val_p_07),
]
base_table = pd.DataFrame(base_rows)
print("06 is included from its saved test submission CSV only.")
base_table

## Step 11: Search validation blend weights for 05.7.1 + 07

In [ ]:
rows = []
for w_0571 in np.round(np.arange(0.0, 1.0 + 1e-9, WEIGHT_STEP), 10):
    w_07 = float(round(1.0 - w_0571, 10))
    val_blend = w_0571 * val_p_0571 + w_07 * val_p_07
    thr, score = best_threshold_score(val_y, val_blend)
    sens, spec = sens_spec(val_y, val_blend, threshold=thr)
    auroc = roc_auc_score(val_y, val_blend)
    rows.append({
        "w_0571": float(w_0571),
        "w_07": float(w_07),
        "thr": float(thr),
        "score": float(score),
        "auroc": float(auroc),
        "sens": float(sens),
        "spec": float(spec),
    })

blend_table = pd.DataFrame(rows).sort_values(["score", "auroc"], ascending=False).reset_index(drop=True)
best_blend = blend_table.iloc[0].to_dict()

print("Best validation blend for the checkpoint-backed models:")
print(best_blend)
blend_table.head(15)

## Step 12: Validation performance and overfitting check

In [ ]:
best_w_0571 = best_blend["w_0571"]
best_w_07 = best_blend["w_07"]
best_thr = best_blend["thr"]

train_blend = best_w_0571 * train_p_0571 + best_w_07 * train_p_07
val_blend = best_w_0571 * val_p_0571 + best_w_07 * val_p_07

train_best = metric_row("train_best_thr", train_y, train_blend)
val_best = metric_row("val_best_thr", val_y, val_blend)
train_at_val_thr = metric_row("train_at_val_thr", train_y, train_blend, threshold=best_thr)
val_at_val_thr = metric_row("val_at_val_thr", val_y, val_blend, threshold=best_thr)

summary_table = pd.DataFrame([train_best, val_best, train_at_val_thr, val_at_val_thr])
summary_table

### Step 12.1: ROC overfitting check

In [ ]:
train_fpr, train_tpr, _ = roc_curve(train_y, train_blend)
val_fpr, val_tpr, _ = roc_curve(val_y, val_blend)
train_auc = roc_auc_score(train_y, train_blend)
val_auc = roc_auc_score(val_y, val_blend)

plt.figure(figsize=(7, 6))
plt.plot(train_fpr, train_tpr, label=f"Train eval AUROC={train_auc:.4f}", linewidth=2)
plt.plot(val_fpr, val_tpr, label=f"Validation AUROC={val_auc:.4f}", linewidth=2)
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Notebook 10 Overfitting Check: Train vs Validation ROC")
plt.legend(loc="lower right")
plt.grid(alpha=0.2)
plt.show()

print(f"Validation-backed weights: 05.7.1={best_w_0571:.2f}, 07={best_w_07:.2f}")
print(f"Validation threshold: {best_thr:.3f}")
print(f"AUROC gap (train - val): {train_auc - val_auc:+.4f}")
print(f"06 is test-only here with manual weight {MANUAL_W_06:.2f} because notebook 06 did not save reusable validation artifacts.")

## Step 13: Write blended submission with 06 added at test time

In [ ]:
remaining = 1.0 - MANUAL_W_06
final_w_0571 = remaining * best_w_0571
final_w_07 = remaining * best_w_07
final_w_06 = MANUAL_W_06

test_blend = final_w_0571 * test_p_0571 + final_w_06 * test_p_06 + final_w_07 * test_p_07

sub = pd.DataFrame({
    "image_file": test_names,
    "prob": test_blend,
})
sub["pred"] = (sub["prob"] >= best_thr).astype(int)

stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M")
out_path = PRED_DIR / f"submission_10_three_model_blend_{stamp}.csv"
sub.to_csv(out_path, index=False)

print(f"Final test weights -> 05.7.1={final_w_0571:.3f}, 06={final_w_06:.3f}, 07={final_w_07:.3f}")
print(f"Wrote {out_path}")
print(f"Positive rate: {sub['pred'].mean():.3f}")
sub.head()